## Python Memory Management

Memory management in Python involves a combination of automatic garbage collection, reference counting, and various internal optimizations to efficiently manage memory allocation and deallocation. Understanding these mechanisms can help developers write more efficient and robust applications.

1. Key Concepts in Python Memory Management
2. Memory allocation and deallocation
3. Reference Counting
4. Garbage Collecting
5. The gc Module
6. Memory Management Best Practices

## Reference Counting

Reference counting is the primary method Python uses to manage memory. Each object in Python maintains a count of references pointing to it. When the reference count drops to zero, the memory occupied by the object is deallocated.

In [1]:
import sys

a = []
print(sys.getrefcount(a))

2


output is 2, because one reference from 'a' and one from getrefcount() methon

In [2]:
b = a
print(sys.getrefcount(b))
print(sys.getrefcount(a))

3
3


In [3]:
del b

try:
    print(sys.getrefcount(b))
except:
    print("There is no more b")

print(sys.getrefcount(a))

There is no more b
2


In [4]:
del a
try:
    print(sys.getrefcount(a))
except:
    print("There is no more a")

There is no more a


#### Garbage Collection

Python includes a cyclic collector to handle reference cycles. Reference cycles occur when objects reference each other, preventing their reference counts from reaching zero.

In [5]:
import gc

## enable garbage collection
gc.enable()

In [6]:
## disable garbage collection
gc.disable()

In [7]:
## manual trigger garbage collection
gc.collect()

0

In [8]:
## get garbage collection stats
print(gc.get_stats())

[{'collections': 200, 'collected': 1585, 'uncollectable': 0}, {'collections': 18, 'collected': 704, 'uncollectable': 0}, {'collections': 2, 'collected': 28, 'uncollectable': 0}]


In [9]:
## get unreachable objects
print(gc.garbage)

[]


#### Memory Management Best Practices

1. Use Local Variables: Local variables have a shorter lifespan and are freed sooner than global variables.
2. Avoid Circular References: Circular references can lead to memory leaks if not properly managed.
3. Use Generators: Generators produce items one at a time and only keep one item in memory at a time, making them memory efficient.
4. Explicitly Delete Objects: Use the del statement to delete variables and objects explicitly.
5. Profile Memory Usage: Use memory profiling tools like tracemalloc and memory_profiler to identify memory leaks and optimize memory usage.

In [10]:
## Handling Circular References
import gc

class MyObject:
    def __init__(self, name):
        self.name = name
        print(f"Object {self.name} created")
        
    def __del__(self):
        print(f"Object {self.name} deleted")
        
## create circular reference
obj1 = MyObject('obj1')
obj2 = MyObject('obj2')

obj1.ref = obj2
obj2.ref = obj1

del obj1
del obj2

## Manually trigger the garbage collection
gc.collect()

## Without writing the code gc.collect(), you will not deallocate the memory due to
## circular reference. So, you will not see the output Object obj1 and obj2 are deleted
## You can try it by commiting the gc.collect()


## You can try it commenting gc.collect() and run it more than 2 times. And uncomment it
## see the output by running.

Object obj1 created
Object obj2 created
Object obj1 deleted
Object obj2 deleted


4

In [11]:
## Print collected objects
try:
    print(f"Garbage collected objects: {gc.garbage}")
except:
    print(f"All the garbage collected are removed")

Garbage collected objects: []


#### Generator For Memory Efficiency

Generators allow you to produce items one at a time, using memory efficiently by only keeping one item in memory at a time.

In [13]:
def generate_numbers(n):
    for i in range(n):
        yield i
        
## using the generator
for num in generate_numbers(10000):
    print(num)
    if num > 10:
        break

0
1
2
3
4
5
6
7
8
9
10
11


#### Profiling Memory Usage with tracemalloc

In [14]:
import tracemalloc

def create_list():
    return [i for i in range(10000)]

def main():
    tracemalloc.start()
    
    create_list()
    
    snapshot = tracemalloc.take_snapshot()
    top_stats = snapshot.statistics('lineno')
    
    print("[ Top 10 ]")
    for stat in top_stats[:10]:
        print(stat)

In [15]:
main()

[ Top 10 ]
C:\Users\BHUPATHE\AppData\Local\Temp\ipykernel_14436\4238193193.py:11: size=416 B, count=1, average=416 B
C:\Users\BHUPATHE\AppData\Local\Temp\ipykernel_14436\4238193193.py:4: size=416 B, count=1, average=416 B


In [16]:
def main():
    tracemalloc.start()
    
    create_list()
    
    snapshot = tracemalloc.take_snapshot()
    top_stats = snapshot.statistics('lineno')
    
    print("[ Top 10 ]")
    for stat in top_stats[::]:
        print(stat)

main()

[ Top 10 ]
d:\Udemy Courses\complete_data_science_machine_learning_dl_nlp_bootcamp\.venv\lib\site-packages\IPython\core\compilerop.py:86: size=5170 B, count=79, average=65 B
C:\Users\BHUPATHE\AppData\Roaming\uv\python\cpython-3.10-windows-x86_64-none\lib\json\decoder.py:353: size=4615 B, count=41, average=113 B
d:\Udemy Courses\complete_data_science_machine_learning_dl_nlp_bootcamp\.venv\lib\site-packages\IPython\core\compilerop.py:174: size=3375 B, count=33, average=102 B
d:\Udemy Courses\complete_data_science_machine_learning_dl_nlp_bootcamp\.venv\lib\site-packages\zmq\sugar\attrsettr.py:45: size=1979 B, count=36, average=55 B
d:\Udemy Courses\complete_data_science_machine_learning_dl_nlp_bootcamp\.venv\lib\site-packages\traitlets\traitlets.py:1521: size=1848 B, count=11, average=168 B
d:\Udemy Courses\complete_data_science_machine_learning_dl_nlp_bootcamp\.venv\lib\site-packages\jupyter_client\session.py:98: size=1489 B, count=8, average=186 B
C:\Users\BHUPATHE\AppData\Roaming\uv\py